In [ ]:
#CELL 1
import torch, os, csv, pandas as pd, numpy as np
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW

# Settings
DATA_ROOT   = '/kaggle/input/datasets/angeliquebreedt/semrel2024-raw/raw'
AUG_DIR     = '/kaggle/working/augmented'
CKPT_DIR    = '/kaggle/working/checkpoints'
RESULTS_LOG = '/kaggle/working/results_log.csv'
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(AUG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
#CELL 2:
def back_translate(sentences, src_lang_model, tgt_lang_model, 
                   src_tokenizer, tgt_tokenizer, batch_size=32):
    """Translate sentences to English then back to source language."""
    results = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        # Translate to English
        inputs = src_tokenizer(batch, return_tensors='pt', 
                               padding=True, truncation=True, 
                               max_length=128).to(DEVICE)
        with torch.no_grad():
            translated = src_lang_model.generate(**inputs)
        english = src_tokenizer.batch_decode(translated, 
                                              skip_special_tokens=True)
        # Translate back to source
        inputs2 = tgt_tokenizer(english, return_tensors='pt',
                                padding=True, truncation=True,
                                max_length=128).to(DEVICE)
        with torch.no_grad():
            back = tgt_lang_model.generate(**inputs2)
        back_translated = tgt_tokenizer.batch_decode(back, 
                                                      skip_special_tokens=True)
        results.extend(back_translated)
        if i % 200 == 0:
            print(f"  Translated {i}/{len(sentences)}...")
    return results

def augment_language(lang_code, src_to_en_model, en_to_src_model,
                     src_to_en_tok, en_to_src_tok, quality_threshold=0.75):
    """Full augmentation pipeline for one language."""
    print(f"\n{'='*50}")
    print(f"Augmenting {lang_code.upper()}")
    print(f"{'='*50}")
    
    df = pd.read_csv(f'{DATA_ROOT}/{lang_code}/train.csv')
    print(f"Original training pairs: {len(df)}")
    
    # Back-translate both sentences
    print("Back-translating sentence1...")
    bt_s1 = back_translate(df['sentence1'].tolist(), 
                           src_to_en_model, en_to_src_model,
                           src_to_en_tok, en_to_src_tok)
    print("Back-translating sentence2...")
    bt_s2 = back_translate(df['sentence2'].tolist(),
                           src_to_en_model, en_to_src_model,
                           src_to_en_tok, en_to_src_tok)
    
    # Quality filtering using cosine similarity
    print("Filtering by quality...")
    sim_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
    
    orig_s1_emb = sim_model.encode(df['sentence1'].tolist(), batch_size=64)
    bt_s1_emb   = sim_model.encode(bt_s1, batch_size=64)
    orig_s2_emb = sim_model.encode(df['sentence2'].tolist(), batch_size=64)
    bt_s2_emb   = sim_model.encode(bt_s2, batch_size=64)
    
    def cos_sim(a, b):
        return np.sum(a*b, axis=1) / (np.linalg.norm(a, axis=1) * 
                                       np.linalg.norm(b, axis=1))
    
    sim_s1 = cos_sim(orig_s1_emb, bt_s1_emb)
    sim_s2 = cos_sim(orig_s2_emb, bt_s2_emb)
    
    mask = (sim_s1 >= quality_threshold) & (sim_s2 >= quality_threshold)
    survival_rate = mask.sum() / len(mask)
    print(f"Pairs generated: {len(df)}")
    print(f"Pairs surviving filter: {mask.sum()} ({survival_rate:.1%})")
    
    aug_df = pd.DataFrame({
        'sentence1': np.array(bt_s1)[mask],
        'sentence2': np.array(bt_s2)[mask],
        'label': df['label'].values[mask]
    })
    
    out_path = f'{AUG_DIR}/{lang_code}_augmented.csv'
    aug_df.to_csv(out_path, index=False)
    print(f"Saved {len(aug_df)} augmented pairs → {out_path}")
    return aug_df, survival_rate

In [ ]:
#CELL 3:
# Use multilingual model - supports Hausa and Kinyarwanda
print("Loading multilingual translation models...")

# mul-en: translates FROM many languages TO English
mul_en_tok = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-mul-en')
mul_en_mdl = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-mul-en').to(DEVICE)

# en-mul: translates FROM English TO many languages  
en_mul_tok = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-mul')
en_mul_mdl = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-mul').to(DEVICE)

print("Models loaded.")

# Test with a Hausa sentence
test_sentences = [">>hau<< Hello, how are you?"]
inputs = en_mul_tok(test_sentences, return_tensors='pt', padding=True).to(DEVICE)
with torch.no_grad():
    out = en_mul_mdl.generate(**inputs)
print("Test translation:", en_mul_tok.decode(out[0], skip_special_tokens=True))

In [ ]:
# CELL 4 
#Load similarity model ONCE outside the function (uses GPU)
print("Loading similarity model...")
sim_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
sim_model = sim_model.to(DEVICE)
print("Ready.")

def back_translate_multilingual(sentences, src_lang_tag,
                                 to_en_model, to_en_tok,
                                 from_en_model, from_en_tok,
                                 batch_size=64):  # increased from 32
    results = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        tagged = [f">>{src_lang_tag}<< {s}" for s in batch]
        inputs = to_en_tok(tagged, return_tensors='pt',
                           padding=True, truncation=True,
                           max_length=64).to(DEVICE)  # reduced from 128
        with torch.no_grad():
            translated = to_en_model.generate(
                **inputs,
                max_new_tokens=64,      # limit output length
                num_beams=1,            # greedy decoding - much faster
                early_stopping=True
            )
        english = to_en_tok.batch_decode(translated, skip_special_tokens=True)
        tagged_back = [f">>{src_lang_tag}<< {s}" for s in english]
        inputs2 = from_en_tok(tagged_back, return_tensors='pt',
                              padding=True, truncation=True,
                              max_length=64).to(DEVICE)
        with torch.no_grad():
            back = from_en_model.generate(
                **inputs2,
                max_new_tokens=64,
                num_beams=1,
                early_stopping=True
            )
        back_translated = from_en_tok.batch_decode(back, skip_special_tokens=True)
        results.extend(back_translated)
        if i % 320 == 0:
            print(f"  Translated {i}/{len(sentences)}...")
    return results

def augment_language_mul(lang_code, lang_tag, quality_threshold=0.75):
    print(f"\n{'='*50}")
    print(f"Augmenting {lang_code.upper()} (tag: {lang_tag})")
    print(f"{'='*50}")
    
    df = pd.read_csv(f'{DATA_ROOT}/{lang_code}/train.csv')
    print(f"Original training pairs: {len(df)}")
    
    print("Back-translating sentence1...")
    bt_s1 = back_translate_multilingual(
        df['sentence1'].tolist(), lang_tag,
        mul_en_mdl, mul_en_tok, en_mul_mdl, en_mul_tok)
    
    print("Back-translating sentence2...")
    bt_s2 = back_translate_multilingual(
        df['sentence2'].tolist(), lang_tag,
        mul_en_mdl, mul_en_tok, en_mul_mdl, en_mul_tok)
    
    # Quality filtering - use GPU encoding
    print("Quality filtering (GPU)...")
    orig_s1 = df['sentence1'].tolist()
    orig_s2 = df['sentence2'].tolist()
    
    # Encode in batches using GPU
    orig_s1_emb = sim_model.encode(orig_s1, batch_size=128, 
                                    show_progress_bar=True,
                                    convert_to_numpy=True)
    bt_s1_emb   = sim_model.encode(bt_s1,   batch_size=128,
                                    show_progress_bar=True,
                                    convert_to_numpy=True)
    orig_s2_emb = sim_model.encode(orig_s2, batch_size=128,
                                    show_progress_bar=True,
                                    convert_to_numpy=True)
    bt_s2_emb   = sim_model.encode(bt_s2,   batch_size=128,
                                    show_progress_bar=True,
                                    convert_to_numpy=True)
    
    def cos_sim(a, b):
        return np.sum(a*b, axis=1) / (
            np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
    
    sim_s1 = cos_sim(orig_s1_emb, bt_s1_emb)
    sim_s2 = cos_sim(orig_s2_emb, bt_s2_emb)
    mask = (sim_s1 >= quality_threshold) & (sim_s2 >= quality_threshold)
    survival_rate = mask.sum() / len(mask)
    
    print(f"Pairs generated: {len(df)}")
    print(f"Pairs surviving filter: {mask.sum()} ({survival_rate:.1%})")
    
    aug_df = pd.DataFrame({
        'sentence1': np.array(bt_s1)[mask],
        'sentence2': np.array(bt_s2)[mask],
        'label':     df['label'].values[mask]
    })
    
    out_path = f'{AUG_DIR}/{lang_code}_augmented.csv'
    aug_df.to_csv(out_path, index=False)
    print(f"Saved {len(aug_df)} pairs → {out_path}")
    return aug_df, survival_rate

print("Functions defined.")

In [ ]:
#CELL 5
hau_aug, hau_rate = augment_language_mul('hau', 'hau')
print(f"\nHausa done. Survival rate: {hau_rate:.1%}") 

In [ ]:
# CELL 6 - rerun with much lower threshold to capture usable pairs
hau_aug, hau_rate = augment_language_mul('hau', 'hau', quality_threshold=0.30)
print(f"\nHausa done. Survival rate: {hau_rate:.1%}")

In [ ]:
#CELL 7
kin_aug, kin_rate = augment_language_mul('kin', 'kin', quality_threshold=0.30)
print(f"\nKinyarwanda done. Survival rate: {kin_rate:.1%}")

In [ ]:
# CELL 7B — XLM-R augmented retraining (AU-1 and AU-3)
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 10
LR = 2e-5
PATIENCE = 3
MODEL_NAME = 'xlm-roberta-base'

class SemRelDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(str(row['sentence1']), str(row['sentence2']),
                             truncation=True, max_length=self.max_len,
                             padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'label': torch.tensor(float(row['label']), dtype=torch.float)}

from transformers import XLMRobertaModel
class SemRelModel(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = XLMRobertaModel.from_pretrained(model_name)
        self.regressor = torch.nn.Linear(self.encoder.config.hidden_size, 1)
        self.dropout = torch.nn.Dropout(0.1)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.regressor(self.dropout(out.last_hidden_state[:,0,:])).squeeze(-1)

def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(batch['input_ids'].to(DEVICE),
                       batch['attention_mask'].to(DEVICE)).cpu().numpy()
            preds.extend(out); labels.extend(batch['label'].numpy())
    rho, _ = spearmanr(preds, labels)
    mse = mean_squared_error(labels, preds)
    return rho, mse, preds

def train_augmented(lang_code, lang_name, tokenizer, aug_df):
    print(f"\n{'='*60}")
    print(f"  Training XLM-R + augmentation on {lang_name}")
    print(f"{'='*60}")
    orig_df  = pd.read_csv(f'{DATA_ROOT}/{lang_code}/train.csv')
    train_df = pd.concat([orig_df, aug_df], ignore_index=True)
    dev_df   = pd.read_csv(f'{DATA_ROOT}/{lang_code}/dev.csv')
    test_df  = pd.read_csv(f'{DATA_ROOT}/{lang_code}/test.csv')
    print(f"  Combined: {len(train_df)} pairs")
    train_loader = DataLoader(SemRelDataset(train_df, tokenizer),
                              batch_size=BATCH_SIZE, shuffle=True)
    dev_loader   = DataLoader(SemRelDataset(dev_df,   tokenizer), batch_size=BATCH_SIZE)
    test_loader  = DataLoader(SemRelDataset(test_df,  tokenizer), batch_size=BATCH_SIZE)
    model     = SemRelModel(MODEL_NAME).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer,
                    num_warmup_steps=int(0.1*total_steps),
                    num_training_steps=total_steps)
    loss_fn = torch.nn.MSELoss()
    best_rho, patience_counter, best_epoch = -1, 0, 0
    for epoch in range(1, EPOCHS+1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(ids, mask), lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
        dev_rho, dev_mse, _ = evaluate(model, dev_loader)
        print(f"  Epoch {epoch:2d} | loss={total_loss/len(train_loader):.4f} | dev ρ={dev_rho:.4f}")
        if dev_rho > best_rho:
            best_rho = dev_rho; best_epoch = epoch; patience_counter = 0
            torch.save(model.state_dict(), f'{CKPT_DIR}/xlmr_{lang_code}_aug_best.pt')
            print(f"           ✓ New best (ρ={best_rho:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}"); break
    model.load_state_dict(torch.load(f'{CKPT_DIR}/xlmr_{lang_code}_aug_best.pt'))
    test_rho, test_mse, _ = evaluate(model, test_loader)
    print(f"\n  Best epoch: {best_epoch} | Test ρ={test_rho:.4f} | MSE={test_mse:.4f}")
    return test_rho, test_mse

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# AU-1: XLM-R + augmented Hausa
rho_au1, mse_au1 = train_augmented('hau', 'Hausa', tokenizer, hau_aug)

# AU-3: XLM-R + augmented Kinyarwanda
rho_au3, mse_au3 = train_augmented('kin', 'Kinyarwanda', tokenizer, kin_aug)

print(f"\nAU-1 Hausa XLM-R+aug:       ρ={rho_au1:.4f}, MSE={mse_au1:.4f}")
print(f"AU-3 Kinyarwanda XLM-R+aug: ρ={rho_au3:.4f}, MSE={mse_au3:.4f}")

In [ ]:
#CELL 8
AFRI_MODEL = 'castorini/afriberta_large'
AFRI_BATCH = 32

class AfriBERTaModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(AFRI_MODEL)
        self.regressor = torch.nn.Linear(self.encoder.config.hidden_size, 1)
        self.dropout = torch.nn.Dropout(0.1)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.regressor(self.dropout(out.last_hidden_state[:,0,:])).squeeze(-1)

def train_augmented_afri(lang_code, lang_name, tokenizer, aug_df):
    print(f"\n{'='*60}")
    print(f"  Training AfriBERTa + augmentation on {lang_name}")
    print(f"{'='*60}")
    
    orig_df = pd.read_csv(f'{DATA_ROOT}/{lang_code}/train.csv')
    train_df = pd.concat([orig_df, aug_df], ignore_index=True)
    dev_df   = pd.read_csv(f'{DATA_ROOT}/{lang_code}/dev.csv')
    test_df  = pd.read_csv(f'{DATA_ROOT}/{lang_code}/test.csv')
    print(f"  Combined: {len(train_df)} pairs")
    
    train_loader = DataLoader(SemRelDataset(train_df, tokenizer, MAX_LEN),
                              batch_size=AFRI_BATCH, shuffle=True)
    dev_loader   = DataLoader(SemRelDataset(dev_df,   tokenizer, MAX_LEN),
                              batch_size=AFRI_BATCH)
    test_loader  = DataLoader(SemRelDataset(test_df,  tokenizer, MAX_LEN),
                              batch_size=AFRI_BATCH)
    
    model     = AfriBERTaModel().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer,
                    num_warmup_steps=int(0.1*total_steps),
                    num_training_steps=total_steps)
    loss_fn = torch.nn.MSELoss()
    best_rho, patience_counter, best_epoch = -1, 0, 0
    
    for epoch in range(1, EPOCHS+1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(ids, mask), lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
        dev_rho, dev_mse, _ = evaluate(model, dev_loader)
        print(f"  Epoch {epoch:2d} | loss={total_loss/len(train_loader):.4f} "
              f"| dev ρ={dev_rho:.4f}")
        if dev_rho > best_rho:
            best_rho = dev_rho; best_epoch = epoch; patience_counter = 0
            torch.save(model.state_dict(),
                      f'{CKPT_DIR}/afriberta_{lang_code}_aug_best.pt')
            print(f"           ✓ New best (ρ={best_rho:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"  Early stopping at epoch {epoch}"); break
    
    model.load_state_dict(torch.load(
        f'{CKPT_DIR}/afriberta_{lang_code}_aug_best.pt'))
    test_rho, test_mse, _ = evaluate(model, test_loader)
    print(f"\n  Best epoch: {best_epoch} | Test ρ={test_rho:.4f} | MSE={test_mse:.4f}")
    return test_rho, test_mse

afri_tokenizer = AutoTokenizer.from_pretrained(AFRI_MODEL)

# AU-2: AfriBERTa + augmented Hausa
rho_au2, mse_au2 = train_augmented_afri('hau', 'Hausa', afri_tokenizer, hau_aug)

# AU-4: AfriBERTa + augmented Kinyarwanda
rho_au4, mse_au4 = train_augmented_afri('kin', 'Kinyarwanda', afri_tokenizer, kin_aug)

print(f"\nAU-2 Hausa AfriBERTa+aug:       ρ={rho_au2:.4f}, MSE={mse_au2:.4f}")
print(f"AU-4 Kinyarwanda AfriBERTa+aug: ρ={rho_au4:.4f}, MSE={mse_au4:.4f}")

In [ ]:
#CELL 9
results = [
    ('AU-1', 'xlmr_augmented',      'xlm-roberta-base',          
     'Hausa',       rho_au1, mse_au1, True,  
     f'XLM-R fine-tuned on original+augmented Hausa. Survival rate: {hau_rate:.1%}'),
    ('AU-2', 'afriberta_augmented', 'castorini/afriberta_large',  
     'Hausa',       rho_au2, mse_au2, True,  
     f'AfriBERTa fine-tuned on original+augmented Hausa. Survival rate: {hau_rate:.1%}'),
    ('AU-3', 'xlmr_augmented',      'xlm-roberta-base',          
     'Kinyarwanda', rho_au3, mse_au3, True,  
     f'XLM-R fine-tuned on original+augmented Kinyarwanda. Survival rate: {kin_rate:.1%}'),
    ('AU-4', 'afriberta_augmented', 'castorini/afriberta_large',  
     'Kinyarwanda', rho_au4, mse_au4, True,  
     f'AfriBERTa fine-tuned on original+augmented Kinyarwanda. Survival rate: {kin_rate:.1%}'),
]

headers = ['experiment_id','model','model_variant','language',
           'split','spearman_rho','mse','augmented','notes']

with open(RESULTS_LOG, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    for r in results:
        writer.writerow({
            'experiment_id': r[0], 'model': r[1], 'model_variant': r[2],
            'language': r[3], 'split': 'test',
            'spearman_rho': round(r[4], 4), 'mse': round(r[5], 4),
            'augmented': r[6], 'notes': r[7]
        })

import pandas as pd
df = pd.read_csv(RESULTS_LOG)
print("\n✓ Augmentation results:")
print(df[['experiment_id','language','spearman_rho','mse']].to_string(index=False))
print("\n✓ Hausa augmentation survival rate:", f"{hau_rate:.1%}")
print("✓ Kinyarwanda augmentation survival rate:", f"{kin_rate:.1%}")
print("\nDownload results_log.csv and augmented CSVs from Output panel.")